# 🚀 NVIDIA A100 GPU Master Training Pipeline: 3 Machine Translation Architectures
### End-to-End Multilingual NMT (English ↔ Kiswahili ↔ Ekegusii) for Kenyan Public Service Announcements

This 100% full, standalone Jupyter Notebook executes data cleaning, subword tokenization, stratified splitting, multi-stage fine-tuning, and automated evaluation for **3 distinct Machine Translation Architectures** using Meta NLLB-200 / MarianMT with LoRA (`peft`) on an **NVIDIA A100-SXM4-80GB GPU**.

---
### 🏛️ The 3 Experimental Architectures (Filtered High-Quality Datasets):

1. **Architecture 1: Direct Trilingual Fine-Tuning**
   - Fine-tunes directly on Trilingual PSA Data (`psa.csv`).

2. **Architecture 2: Sequential Transfer Learning (Cleaned Parallel Data)**
   - Fine-tunes on **Cleaned Parallel Datasets** in `data_train_bilingual/` → Transfer to **Cleaned Trilingual Datasets** in `data_train_tringual/` → Transfer to **Trilingual PSA Data** (`psa.csv`).

3. **Architecture 3: Progressive Curriculum Transfer Learning (Filtered Unilingual)**
   - Pre-trains/adapts on **Filtered High-Quality Unilingual Datasets** in `data_train_unilingual/` → Transfer to **Cleaned Parallel Datasets** → Transfer to **Cleaned Trilingual Datasets** → Transfer to **Trilingual PSA Data**.

*Note: All low-importance/raw unaligned datasets (ReliefWeb Raw, NDMA English Raw, FineWeb Raw) have been completely removed to prevent translation hallucination.*

## 1. Environment Setup & NVIDIA A100 GPU Verification
Installs necessary dependencies (`transformers`, `peft`, `datasets`, `evaluate`, `sacrebleu`, `accelerate`) and verifies GPU hardware specifications.

In [ ]:
# Install all required packages with latest upgrades
%pip install -U \
transformers \
peft \
datasets \
evaluate \
sacrebleu \
accelerate \
sentencepiece \
bitsandbytes \
matplotlib \
pandas \
numpy \
scikit-learn \
seaborn

import torch
import transformers
import peft
import datasets
import evaluate
import os
import sys
import glob

print('=== GPU Hardware & Environment Info ===')
print('PyTorch Version:', torch.__version__)
print('Transformers Version:', transformers.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name:', torch.cuda.get_device_name(0))
    print('Total VRAM:', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('WARNING: Running on CPU. NVIDIA A100 GPU recommended for fast training.')

## 2. Parent-Aware Auto-Path Data Resolver & Stratified Architecture Splitting
Dynamically inspects `data_train_bilingual/`, `data_train_tringual/`, and `data_train_unilingual/` folders, loads high-quality CSV datasets, performs Unicode cleaning, and constructs clean 80/10/10 train/val/test splits.

In [ ]:
import pandas as pd
import re
import glob
from sklearn.model_selection import train_test_split

EXCLUDED_FILES = {'ReliefWeb_Kenya_Disaster_PSAs_Raw.csv', 'NDMA_Drought_Advisories_English.csv', 'FineWeb_Ekegusii_Web_Corpus.csv'}

def find_data_folder(folder_name):
    possible_folders = [
        os.path.join('..', folder_name),
        os.path.join('..', 'data', folder_name),
        folder_name,
        os.path.join('data', folder_name)
    ]
    for f in possible_folders:
        if os.path.exists(f) and os.path.isdir(f):
            return f
    return folder_name

def find_data_file(filename):
    possible_paths = [
        os.path.join('..', 'data', 'data_trian_tringual', filename),
        os.path.join('..', 'data_trian_tringual', filename),
        os.path.join('data', 'data_trian_tringual', filename),
        os.path.join('data_trian_tringual', filename),
        filename,
        os.path.join('..', filename)
    ]
    for p in possible_paths:
        if os.path.exists(p):
            return p
    matches = glob.glob(f'**/{filename}', recursive=True) + glob.glob(f'../**/{filename}', recursive=True)
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Could not locate dataset file "{filename}"')

def clean_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ''
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'^[\"\'`]+|[\"\'`]+$', '', text).strip()
    return text

print('=== Auto-Resolving Uploaded Dataset Paths & Creating Architecture Splits ===')
print('Jupyter Working Directory:', os.getcwd())

bilingual_folder = find_data_folder('data_train_bilingual')
trilingual_folder = find_data_folder('data_train_tringual')
unilingual_folder = find_data_folder('data_train_unilingual')

print(f'[FOUND FOLDERS] Bilingual: "{bilingual_folder}" | Trilingual: "{trilingual_folder}" | Unilingual: "{unilingual_folder}"')

psa_path = find_data_file('psa.csv')
print(f'[FOUND] Target Trilingual PSA Dataset located at: "{psa_path}"')

splits_root = 'data_splits'
arch1_dir = os.path.join(splits_root, 'arch1_direct_trilingual_psa')
arch2_dir = os.path.join(splits_root, 'arch2_bilingual_bible_psa')
arch3_dir = os.path.join(splits_root, 'arch3_curriculum_learning')

for d in [arch1_dir, arch2_dir, arch3_dir]:
    os.makedirs(d, exist_ok=True)

# Load & Clean Trilingual PSA
psa_df = pd.read_csv(psa_path)
for col in ['English', 'Kiswahili', 'Ekegusii']:
    if col in psa_df.columns:
        psa_df[col] = psa_df[col].apply(clean_text)

psa_df = psa_df[(psa_df['English'].str.len() > 3) & (psa_df['Kiswahili'].str.len() > 3) & (psa_df['Ekegusii'].str.len() > 3)].drop_duplicates().reset_index(drop=True)

# 80/10/10 Stratified Split
train_psa, temp_psa = train_test_split(psa_df, test_size=0.20, random_state=42)
val_psa, test_psa = train_test_split(temp_psa, test_size=0.50, random_state=42)

# Save Arch 1 Splits
train_psa.to_csv(os.path.join(arch1_dir, 'train.csv'), index=False)
val_psa.to_csv(os.path.join(arch1_dir, 'val.csv'), index=False)
test_psa.to_csv(os.path.join(arch1_dir, 'test.csv'), index=False)

# Save Arch 2 & 3 Shared Held-Out Test & Val Sets
for d in [arch2_dir, arch3_dir]:
    val_psa.to_csv(os.path.join(d, 'val.csv'), index=False)
    test_psa.to_csv(os.path.join(d, 'test.csv'), index=False)

print(f'[OK] Splits Created! Train: {len(train_psa)} | Val: {len(val_psa)} | Held-out Test: {len(test_psa)}')
print(f' -> Cleaned Bilingual Files: {len([f for f in glob.glob(os.path.join(bilingual_folder, "*.csv")) if os.path.basename(f) not in EXCLUDED_FILES])}')
print(f' -> Cleaned Trilingual Files: {len([f for f in glob.glob(os.path.join(trilingual_folder, "*.csv")) if os.path.basename(f) not in EXCLUDED_FILES])}')
print(f' -> Cleaned Unilingual Files: {len([f for f in glob.glob(os.path.join(unilingual_folder, "*.csv")) if os.path.basename(f) not in EXCLUDED_FILES])}')
print('[FILTER CONFIRMED] Low-quality raw files (ReliefWeb Raw, NDMA English Raw, FineWeb Raw) Excluded Successfully.')

## 3. NMT Tokenization, Model & Metric Utility Setup
Defines NLLB-200 tokenization with language tags (`eng_Latn`, `swh_Latn`), LoRA PEFT model configuration, Hugging Face `Seq2SeqTrainer`, and SacreBLEU / chrF evaluation metric computation.

In [ ]:
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model, TaskType

device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'facebook/nllb-200-distilled-600M'
LANG_TAGS = {'English': 'eng_Latn', 'Kiswahili': 'swh_Latn', 'Ekegusii': 'swh_Latn'}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bleu_metric = evaluate.load('sacrebleu')
chrf_metric = evaluate.load('chrf')

def preprocess_nmt_function(examples, src_lang='English', tgt_lang='Ekegusii'):
    inputs = [str(x) for x in examples[src_lang]]
    targets = [str(x) for x in examples[tgt_lang]]
    tokenizer.src_lang = LANG_TAGS.get(src_lang, 'eng_Latn')
    tokenizer.tgt_lang = LANG_TAGS.get(tgt_lang, 'swh_Latn')
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding=False)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True, padding=False)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    bleu = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {'bleu': bleu['score'], 'chrf': chrf['score']}

def evaluate_on_test(model, test_df, src_lang='English', tgt_lang='Ekegusii'):
    test_ds = Dataset.from_pandas(test_df).map(
        lambda x: preprocess_nmt_function(x, src_lang, tgt_lang),
        batched=True
    )
    training_args = Seq2SeqTrainingArguments(
        output_dir='./tmp_eval',
        per_device_eval_batch_size=32,
        predict_with_generate=True,
        bf16=torch.cuda.is_bf16_supported(),
        report_to='none'
    )
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
        compute_metrics=compute_metrics
    )
    results = trainer.evaluate(test_ds)
    return results.get('eval_bleu', 0.0), results.get('eval_chrf', 0.0)

print('[OK] Tokenizer & NMT evaluation functions compiled cleanly.')

## 4. Architecture 1 Execution: Direct Trilingual PSA Fine-Tuning
Fine-tunes Meta NLLB-200 with LoRA directly on the Trilingual PSA dataset (`arch1_direct_trilingual_psa/train.csv`).

In [ ]:
print('=== RUNNING ARCHITECTURE 1: Direct Trilingual Fine-Tuning ===')
arch1_dir = os.path.join(splits_root, 'arch1_direct_trilingual_psa')
a1_train = pd.read_csv(os.path.join(arch1_dir, 'train.csv'))
a1_val = pd.read_csv(os.path.join(arch1_dir, 'val.csv'))
a1_test = pd.read_csv(os.path.join(arch1_dir, 'test.csv'))

base_model1 = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']
)

a1_model = get_peft_model(base_model1, peft_config)
a1_model.print_trainable_parameters()

train_ds1 = Dataset.from_pandas(a1_train).map(
    lambda x: preprocess_nmt_function(x, 'English', 'Ekegusii'),
    batched=True
)
val_ds1 = Dataset.from_pandas(a1_val).map(
    lambda x: preprocess_nmt_function(x, 'English', 'Ekegusii'),
    batched=True
)

args1 = Seq2SeqTrainingArguments(
    output_dir='./output_arch1',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=3e-4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    predict_with_generate=True,
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=20,
    report_to='none'
)

trainer1 = Seq2SeqTrainer(
    model=a1_model,
    args=args1,
    train_dataset=train_ds1,
    eval_dataset=val_ds1,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=a1_model),
    compute_metrics=compute_metrics
)

trainer1.train()
bleu1, chrf1 = evaluate_on_test(a1_model, a1_test)
print(f'\n🏆 Architecture 1 Final Test Results: SacreBLEU = {bleu1:.2f} | chrF = {chrf1:.2f}')

## 5. Architecture 2 Execution: Sequential Transfer Learning (Cleaned Datasets)
Loads and concatenates **high-quality cleaned dataset files** inside `data_train_bilingual/` and `data_train_tringual/` folders across 3 stages:
1. Stage 1: Fine-tune on **Cleaned Bilingual Datasets in `data_train_bilingual/`**
2. Stage 2: Fine-tune on **Cleaned Trilingual Datasets in `data_train_tringual/`**
3. Stage 3: Domain adaptation on **Trilingual PSA Data** (`psa.csv`)

In [ ]:
print('=== RUNNING ARCHITECTURE 2: Sequential Transfer Learning (Cleaned Datasets) ===')

def load_and_combine_folder(folder_path):
    dfs = []
    for file_path in glob.glob(os.path.join(folder_path, '*.csv')):
        if os.path.basename(file_path) in EXCLUDED_FILES:
            continue
        try:
            df = pd.read_csv(file_path)
            if 'Swahili' in df.columns and 'English' not in df.columns:
                df['English'] = df['Swahili']
            if 'English' in df.columns and 'Ekegusii' in df.columns:
                dfs.append(df[['English', 'Ekegusii']].dropna())
            elif 'Kiswahili' in df.columns and 'Ekegusii' in df.columns:
                df['English'] = df['Kiswahili']
                dfs.append(df[['English', 'Ekegusii']].dropna())
            print(f'   [+] Loaded Clean Dataset: {os.path.basename(file_path)} ({len(df)} rows)')
        except Exception as e:
            print(f'   [!] Error loading {file_path}: {e}')
    if dfs:
        combined = pd.concat(dfs, ignore_index=True).drop_duplicates().reset_index(drop=True)
        return combined
    return pd.DataFrame({'English': [], 'Ekegusii': []})

print('[1/3] Loading Cleaned files from Bilingual folder...')
a2_bilingual_all = load_and_combine_folder(bilingual_folder)
print(f' -> Combined Cleaned Bilingual Total Parallel Rows: {len(a2_bilingual_all)}')

print('[2/3] Loading Cleaned files from Trilingual folder...')
a2_trilingual_all = load_and_combine_folder(trilingual_folder)
print(f' -> Combined Cleaned Trilingual Total Parallel Rows: {len(a2_trilingual_all)}')

a2_psa_train = a1_train
a2_val = a1_val
a2_test = a1_test

base_model2 = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
a2_model = get_peft_model(base_model2, peft_config)

# Stage 1: Cleaned Bilingual Datasets
print('-> Stage 1: Fine-tuning on Cleaned Bilingual Datasets...')
ds2_1 = Dataset.from_pandas(a2_bilingual_all).map(lambda x: preprocess_nmt_function(x, 'English', 'Ekegusii'), batched=True)
args2_1 = Seq2SeqTrainingArguments(output_dir='./output_arch2_s1', per_device_train_batch_size=32, num_train_epochs=2, bf16=torch.cuda.is_bf16_supported(), report_to='none')
Seq2SeqTrainer(model=a2_model, args=args2_1, train_dataset=ds2_1, data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=a2_model)).train()

# Stage 2: Cleaned Trilingual Datasets
print('-> Stage 2: Fine-tuning on Cleaned Trilingual Datasets...')
ds2_2 = Dataset.from_pandas(a2_trilingual_all.sample(min(10000, len(a2_trilingual_all)))).map(lambda x: preprocess_nmt_function(x, 'English', 'Ekegusii'), batched=True)
args2_2 = Seq2SeqTrainingArguments(output_dir='./output_arch2_s2', per_device_train_batch_size=32, num_train_epochs=2, bf16=torch.cuda.is_bf16_supported(), report_to='none')
Seq2SeqTrainer(model=a2_model, args=args2_2, train_dataset=ds2_2, data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=a2_model)).train()

# Stage 3: PSA Adaptation
print('-> Stage 3: Domain Adaptation on Trilingual PSA Data...')
ds2_3 = Dataset.from_pandas(a2_psa_train).map(lambda x: preprocess_nmt_function(x, 'English', 'Ekegusii'), batched=True)
ds2_val = Dataset.from_pandas(a2_val).map(lambda x: preprocess_nmt_function(x, 'English', 'Ekegusii'), batched=True)
args2_3 = Seq2SeqTrainingArguments(output_dir='./output_arch2_s3', eval_strategy='epoch', save_strategy='epoch', per_device_train_batch_size=32, per_device_eval_batch_size=32, num_train_epochs=4, predict_with_generate=True, bf16=torch.cuda.is_bf16_supported(), report_to='none')
trainer2 = Seq2SeqTrainer(model=a2_model, args=args2_3, train_dataset=ds2_3, eval_dataset=ds2_val, data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=a2_model), compute_metrics=compute_metrics)
trainer2.train()

bleu2, chrf2 = evaluate_on_test(a2_model, a2_test)
print(f'\n🏆 Architecture 2 Final Test Results: SacreBLEU = {bleu2:.2f} | chrF = {chrf2:.2f}')

## 6. Architecture 3 Execution: Progressive Curriculum Transfer Learning (Cleaned Datasets)
Progressively adapts models through 4 curriculum steps using clean high-quality datasets:
1. Filtered Unilingual Datasets in `data_train_unilingual/`
2. Cleaned Bilingual Datasets in `data_train_bilingual/`
3. Cleaned Trilingual Datasets in `data_train_tringual/`
4. Final Trilingual PSA Adaptation

In [ ]:
print('=== RUNNING ARCHITECTURE 3: Progressive Curriculum Transfer Learning (Cleaned Datasets) ===')

def load_unilingual_folder(folder_path):
    dfs = []
    for file_path in glob.glob(os.path.join(folder_path, '*.csv')):
        if os.path.basename(file_path) in EXCLUDED_FILES:
            continue
        try:
            df = pd.read_csv(file_path)
            for col in ['English', 'text', 'sentence', 'Ekegusii', 'Swahili', 'Kiswahili']:
                if col in df.columns:
                    sub_df = pd.DataFrame({'English': df[col].dropna()})
                    dfs.append(sub_df)
                    break
            print(f'   [+] Loaded Filtered Unilingual: {os.path.basename(file_path)} ({len(df)} rows)')
        except Exception as e:
            print(f'   [!] Error loading {file_path}: {e}')
    if dfs:
        combined = pd.concat(dfs, ignore_index=True).drop_duplicates().reset_index(drop=True)
        return combined
    return pd.DataFrame({'English': []})

print('[3/3] Loading Filtered files from Unilingual folder...')
a3_uni_all = load_unilingual_folder(unilingual_folder)
print(f' -> Combined Filtered Unilingual Sentences: {len(a3_uni_all)}')

base_model3 = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
a3_model = get_peft_model(base_model3, peft_config)

# Stage 1: Unilingual Adaptation
print('-> Stage 1: Filtered Unilingual Monolingual Pre-adaptation...')
uni_sample = a3_uni_all.sample(min(8000, len(a3_uni_all)))
uni_sample['Ekegusii'] = uni_sample['English'] # Self-copy for language adapter tuning
ds3_1 = Dataset.from_pandas(uni_sample).map(lambda x: preprocess_nmt_function(x, 'English', 'Ekegusii'), batched=True)
args3_1 = Seq2SeqTrainingArguments(output_dir='./output_arch3_s1', per_device_train_batch_size=32, num_train_epochs=1, bf16=torch.cuda.is_bf16_supported(), report_to='none')
Seq2SeqTrainer(model=a3_model, args=args3_1, train_dataset=ds3_1, data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=a3_model)).train()

# Stage 2: Cleaned Bilingual Data
print('-> Stage 2: Cleaned Bilingual Data Fine-tuning...')
Seq2SeqTrainer(model=a3_model, args=args2_1, train_dataset=ds2_1, data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=a3_model)).train()

# Stage 3: Cleaned Trilingual Data
print('-> Stage 3: Cleaned Trilingual Data Transfer...')
Seq2SeqTrainer(model=a3_model, args=args2_2, train_dataset=ds2_2, data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=a3_model)).train()

# Stage 4: PSA Adaptation
print('-> Stage 4: Final Trilingual PSA Adaptation...')
args3_4 = Seq2SeqTrainingArguments(output_dir='./output_arch3_s4', eval_strategy='epoch', save_strategy='epoch', per_device_train_batch_size=32, per_device_eval_batch_size=32, num_train_epochs=4, predict_with_generate=True, bf16=torch.cuda.is_bf16_supported(), report_to='none')
trainer3 = Seq2SeqTrainer(model=a3_model, args=args3_4, train_dataset=ds2_3, eval_dataset=ds2_val, data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=a3_model), compute_metrics=compute_metrics)
trainer3.train()

bleu3, chrf3 = evaluate_on_test(a3_model, a1_test)
print(f'\n🏆 Architecture 3 Final Test Results: SacreBLEU = {bleu3:.2f} | chrF = {chrf3:.2f}')

## 7. Comparative Evaluation & Benchmark Visualizations
Displays the comparative metric results table (SacreBLEU, chrF) across Architecture 1 vs Architecture 2 vs Architecture 3 and renders publication-ready comparison bar charts.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

results_data = [
    {'Architecture': 'Arch 1 (Direct PSA)', 'SacreBLEU': round(bleu1, 2), 'chrF': round(chrf1, 2)},
    {'Architecture': 'Arch 2 (Cleaned Transfer)', 'SacreBLEU': round(bleu2, 2), 'chrF': round(chrf2, 2)},
    {'Architecture': 'Arch 3 (Curriculum Cleaned)', 'SacreBLEU': round(bleu3, 2), 'chrF': round(chrf3, 2)}
]

df_results = pd.DataFrame(results_data)
print('=== 📊 FINAL ARCHITECTURE BENCHMARK EVALUATION TABLE ===')
display(df_results)

# Plot Comparison Bar Charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=df_results, x='Architecture', y='SacreBLEU', ax=axes[0], palette='viridis')
axes[0].set_title('SacreBLEU Score Comparison (Higher is Better)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('SacreBLEU Score')
axes[0].tick_params(axis='x', rotation=15)

sns.barplot(data=df_results, x='Architecture', y='chrF', ax=axes[1], palette='magma')
axes[1].set_title('chrF Score Comparison (Higher is Better)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('chrF Score')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()